# 01 — Financial primitives

**Code under test:** `fleximorpv2/utils/financial.py`

**Purpose:** Confirm NPV, IRR, LCOE, and payback match hand calculations before trusting pipeline results.

Run cells **top to bottom**. Each markdown cell explains the **next code cell** — what it does, what to inspect in the output, and what counts as a pass.

Track overall audit progress in Obsidian (`FlexiMORP Calculation Audit.md`). These notebooks are the lab workbook, not the checklist.

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("error", category=RuntimeWarning)


def _repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "fleximorpv2").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find fleximorp-project root. "
        "Open Jupyter from the repo or a notebooks/ subdirectory."
    )


_repo = _repo_root()
_audit = _repo / "notebooks" / "audit"
for path in (_repo, _audit):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from _audit_helpers import (
    REPO_ROOT,
    OUTPUT_DIR,
    SITE_OUTPUT_DIR,
    reload_fleximorp,
    assert_close,
    assert_energy_balance,
    assert_cf_bounds,
    assert_positive,
)

reload_fleximorp()
np.random.seed(42)
print(f"Repo root: {REPO_ROOT}")
print(f"Audit outputs: {OUTPUT_DIR}")
print(f"Site outputs: {SITE_OUTPUT_DIR}")

## Step 1 — Golden-case LCOE

**Run the next cell** after filling in `EXPECTED` from your hand calculation.

Use this toy project (flat inputs, no tax complexity):

| Input | Value |
|-------|-------|
| CAPEX | £1,000,000 |
| OPEX | £50,000 / yr (flat) |
| Annual energy | 10,000 MWh / yr (flat, no degradation first) |
| Discount rate | 8% |
| Project life | 20 years |

**Formula:** LCOE = (CAPEX + Σ OPEX/(1+r)^t) / Σ Energy/(1+r)^t

**Pass if:** `assert_close` within 0.01% of your spreadsheet.

**Then:** repeat with degradation enabled in `calculate_lcoe` (hard-coded 0.5%/yr) — LCOE should increase slightly.

**Also inspect:** `calculate_metrics()` — if `annual_energy` is omitted it guesses from `revenue / 0.1`. List any callers that hit this path.

In [ ]:
from fleximorpv2.config import load_config
from fleximorpv2.utils.financial import FinancialCalculator

config = load_config("alaska")
calc = FinancialCalculator(config)

# Hand-compute LCOE for the table above, then set:
EXPECTED = None  # £/MWh

capex = 1_000_000
opex = 50_000
annual_energy = 10_000  # MWh/yr
discount_rate = 0.08
project_life = 20

lcoe = calc.calculate_lcoe(capex, opex, annual_energy, discount_rate, project_life)

if EXPECTED is None:
    raise ValueError("Set EXPECTED from your hand calculation, then re-run")
assert_close(lcoe, EXPECTED, label="LCOE golden case")
print(f"LCOE = {lcoe:.2f} £/MWh")

## Step 2 — NPV and IRR on a 3-year toy cash flow

**Run the next cell** with a simple cash flow you can replicate in Excel (`=NPV`, `=IRR`).

**Pass if:** NPV sign is correct (−CAPEX at year 0, inflows discounted from year 1) and IRR within 0.1 pp of spreadsheet.

In [ ]:
cash_flows = np.array([120_000, 130_000, 140_000])  # years 1–3
initial_investment = 300_000
discount_rate = 0.08

npv = calc.calculate_npv(cash_flows, discount_rate, initial_investment)
irr = calc.calculate_irr(cash_flows, initial_investment)

print(f"NPV = {npv:,.0f}")
print(f"IRR = {irr:.2%}")
# TODO: set EXPECTED_NPV / EXPECTED_IRR from spreadsheet and assert_close

## Step 3 — Sensitivity (direction checks)

**Run the next cell** to plot LCOE vs CAPEX and vs annual energy.

**Pass if:** LCOE rises when CAPEX rises or energy falls (monotonic, no sign inversions).

In [ ]:
capex_range = np.linspace(0.5e6, 2.0e6, 20)
lcoe_vs_capex = [
    calc.calculate_lcoe(c, 50_000, 10_000, 0.08, 20) for c in capex_range
]

energy_range = np.linspace(5_000, 20_000, 20)
lcoe_vs_energy = [
    calc.calculate_lcoe(1e6, 50_000, e, 0.08, 20) for e in energy_range
]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(capex_range / 1e6, lcoe_vs_capex)
axes[0].set(xlabel="CAPEX (£M)", ylabel="LCOE (£/MWh)", title="Higher CAPEX → higher LCOE")
axes[1].plot(energy_range, lcoe_vs_energy)
axes[1].set(xlabel="Annual energy (MWh/yr)", ylabel="LCOE (£/MWh)", title="More energy → lower LCOE")
plt.tight_layout()
plt.show()